## **Evaluation of Social Media Cleaning Method**

In [1]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

In [2]:
spark = sparknlp.start()

:: loading settings :: url = jar:file:/uufs/chpc.utah.edu/common/home/u1332544/my-python-venvs/spark-nlp/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/cache
The jars for the packages stored in: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a726c6b2-b31b-4b6a-a192-2812b2f263f9;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in centr

26/04/25 22:25:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/25 22:25:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Load Data

In [3]:
CLEAN_DATA_PATH = '../cleaned_data/social_media_cleaned_tweets.csv'

In [4]:
df = spark.read.option("header", True).csv(CLEAN_DATA_PATH)
df.show(5)
print("Number of rows:", df.count())

+----+-----------+------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|
+----+-----------+------------+--------------------+
|2401|Borderlands|    Positive|I am coming to th...|
|2401|Borderlands|    Positive|im getting on bor...|
|2401|Borderlands|    Positive|im coming on bord...|
|2401|Borderlands|    Positive|im getting on bor...|
|2401|Borderlands|    Positive|im getting into b...|
+----+-----------+------------+--------------------+
only showing top 5 rows

Number of rows: 74681


### Drop Nulls and Empty Values

In [5]:
df = df.dropna(subset=["cleaned_text"])
print("Number of rows after dropping nulls:", df.count())

Number of rows after dropping nulls: 73700


In [6]:
from pyspark.sql.functions import trim

df = df.filter(trim(col("cleaned_text")) != "")
print("Number of rows after dropping empty strings:", df.count())

Number of rows after dropping empty strings: 73700


### Drop Rows with "Irrelevant" OG Sentiment
These must be dropped as the model can only classify "positive", "neutral", or "negative". 

In [7]:
df = df.filter(col("og_sentiment") != "Irrelevant")
print("Number of rows after dropping irrelevant rows:", df.count())

Number of rows after dropping irrelevant rows: 60874


### Run Model

In [8]:
document_assembler = DocumentAssembler() \
    .setInputCol('cleaned_text') \
    .setOutputCol('document')

tokenizer = Tokenizer() \
    .setInputCols(['document']) \
    .setOutputCol('token')

classifier = XlmRoBertaForSequenceClassification.pretrained('twitter_xlm_roberta_base_sentiment')\
  .setInputCols(["document",'token'])\
  .setOutputCol("pred_sentiment")

pipeline = Pipeline(stages=[
    document_assembler, 
    tokenizer,
    classifier    
])

model = pipeline.fit(df)
result = model.transform(df)

twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
[ | ]26/04/25 22:25:17 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/04/25 22:25:17 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
[ | ]Download done! Loading the resource.
[OK!]


In [9]:
result.show()

[Stage 16:>                                                         (0 + 1) / 1]

Using CPUs
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 49...|[{token, 0, 0, I,...|[{category, 0, 49...|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 48...|[{token, 0, 1, im...|[{category, 0, 48...|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 55...|[{token, 0, 1, im...|[{category, 0, 55...|
|2401|Borderlands|    Positive|im getting into b...|[{document, 0, 51...|[{token, 0, 1, im...|[{category, 0, 51...|
|2402|Borderlands|    Positive|So I spent a few ...|[{documen

In [10]:
# View the actual labels from the pred_sentiment column
result.select("og_sentiment", "cleaned_text", "pred_sentiment.result").show()

[Stage 17:>                                                         (0 + 1) / 1]

+------------+--------------------+----------+
|og_sentiment|        cleaned_text|    result|
+------------+--------------------+----------+
|    Positive|I am coming to th...|[negative]|
|    Positive|im getting on bor...| [neutral]|
|    Positive|im coming on bord...|[negative]|
|    Positive|im getting on bor...| [neutral]|
|    Positive|im getting into b...|[negative]|
|    Positive|So I spent a few ...| [neutral]|
|    Positive|So I spent a coup...| [neutral]|
|    Positive|So I spent a few ...|[positive]|
|    Positive|So I spent a few ...| [neutral]|
|    Positive|2010 So I spent a...| [neutral]|
|    Positive|                 was|[negative]|
|     Neutral|RockHard La Varlo...| [neutral]|
|     Neutral|RockHard La Varlo...|[positive]|
|     Neutral|RockHard La Varlo...| [neutral]|
|     Neutral|RockHard La Vita ...| [neutral]|
|     Neutral|Live Rock Hard mu...|[positive]|
|     Neutral|Izard like me RAR...| [neutral]|
|    Positive|that was the firs...|[positive]|
|    Positive

In [11]:
# Create column with the predicted labels
result = result.withColumn("pred_label",col("pred_sentiment")[0].result)
result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 49...|[{token, 0, 0, I,...|[{category, 0, 49...|  negative|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 48...|[{token, 0, 1, im...|[{category, 0, 48...|   neutral|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|  negative|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 55...|[{token, 0, 1, im...|[{category, 0, 55...|   neutral|
|2401|Borderlands|    Positive|im getting into b...|[{document, 0, 51...|[{token, 0, 1, im...|[{category, 0, 51

### Normalize Label Columns

In [12]:
from pyspark.sql.functions import lower

result = result.withColumn("og_label", lower(col("og_sentiment"))) \
               .withColumn("pred_label", lower(col("pred_label")))

result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|og_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 49...|[{token, 0, 0, I,...|[{category, 0, 49...|  negative|positive|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 48...|[{token, 0, 1, im...|[{category, 0, 48...|   neutral|positive|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|  negative|positive|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 55...|[{token, 0, 1, im...|[{category, 0, 55...|   neutral|positive|
|2401|Borderlands|    Positive|im getting into b

### Calculate Accuracy

In [13]:
eval_df = result.select("og_label", "pred_label").toPandas()
eval_df

,og_label,pred_label
0,positive,negative
1,positive,neutral
2,positive,negative
3,positive,neutral
4,positive,negative
...,...,...
60869,positive,negative
60870,positive,negative
60871,positive,negative
60872,positive,neutral


In [14]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(eval_df["og_label"], eval_df["pred_label"])
print("Accuracy:", accuracy)

Accuracy: 0.4241383842034366


### Calculate F1 Scores

In [15]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="macro")
print("Macro F1:", macro_f1)

weighted_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="weighted")
print("Weighted F1:", weighted_f1)

Macro F1: 0.4179798946649565
Weighted F1: 0.41541235317569414


### Classification Report

In [16]:
from sklearn.metrics import classification_report

print(classification_report(eval_df["og_label"], eval_df["pred_label"]))

              precision    recall  f1-score   support

    negative       0.64      0.26      0.37     22276
     neutral       0.33      0.69      0.44     18007
    positive       0.55      0.37      0.44     20591

    accuracy                           0.42     60874
   macro avg       0.51      0.44      0.42     60874
weighted avg       0.52      0.42      0.42     60874

